# ICT 110 – Task 1: Workflow A (Notebook)

**Notebook questions required (N): A2, A3, A4, and A7.**

Project: **Predicting Students’ Dropout and Academic Success**

## A2. Type of machine learning problem

This is a **supervised multiclass classification** problem. The target variable, `Target`, contains three predefined categories: `Dropout`, `Enrolled`, and `Graduate`. It is classification because the task is to predict one of these categories.

## Load the dataset using a direct link

Original dataset: [UCI Machine Learning Repository – Dataset 697](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

The original dataset is published by UCI. Because the UCI download endpoint returned an expired SSL certificate in Colab, the code reads a GitHub-hosted CSV copy of this dataset through a direct Raw URL. The dataset remains credited to UCI.

In [ ]:
# Import pandas
import pandas as pd

# Load the CSV directly from the Raw URL
DATASET_URL = "https://raw.githubusercontent.com/alecngo/Predict-students-dropout-and-academic-success/refs/heads/master/data.csv"

df = pd.read_csv(DATASET_URL, sep=';')

# Remove any extra spaces or tabs from column names
df.columns = df.columns.str.strip()

# Define the target labels
class_labels = ['Dropout', 'Enrolled', 'Graduate']

# Check that the expected target column is present
if 'Target' not in df.columns:
    raise ValueError("The downloaded CSV does not contain a 'Target' column.")

# Check that the expected target labels are present
observed_labels = set(df['Target'].dropna().unique())

if observed_labels != set(class_labels):
    raise ValueError(
        f"Unexpected target labels: {sorted(observed_labels)}"
    )

# Display dataset information
print("Dataset shape:", df.shape)
print("Number of student records:", len(df))
print("Number of predictor features:", df.shape[1] - 1)

# Display exactly 500 rows continuously
print("\nFirst 500 student records:\n")

print(df.head(500).to_string(index=True))

Dataset shape: (4424, 37)
Number of student records: 4424
Number of predictor features: 36

First 500 student records:

     Marital status  Application mode  Application order  Course  Daytime/evening attendance  Previous qualification  Previous qualification (grade)  Nacionality  Mother's qualification  Father's qualification  Mother's occupation  Father's occupation  Admission grade  Displaced  Educational special needs  Debtor  Tuition fees up to date  Gender  Scholarship holder  Age at enrollment  International  Curricular units 1st sem (credited)  Curricular units 1st sem (enrolled)  Curricular units 1st sem (evaluations)  Curricular units 1st sem (approved)  Curricular units 1st sem (grade)  Curricular units 1st sem (without evaluations)  Curricular units 2nd sem (credited)  Curricular units 2nd sem (enrolled)  Curricular units 2nd sem (evaluations)  Curricular units 2nd sem (approved)  Curricular units 2nd sem (grade)  Curricular units 2nd sem (without evaluations)  Unemploymen

## A3. Target variable and labels

The target variable is **`Target`**. It contains the students’ academic outcome at the end of the normal duration of the course. The three labels are:

- `Dropout`
- `Enrolled`
- `Graduate`

The project uses information available at enrollment and at the end of the first and second semesters. Predictions are made after the second semester to estimate the students’ eventual academic outcomes.

In [ ]:
# Count each target label and calculate its percentage
# class_labels was defined in the dataset-loading cell above.
target_summary = (
    df['Target']
    .value_counts()
    .reindex(class_labels)
    .rename_axis('Target')
    .to_frame('Count')
)

target_summary['Percentage'] = (
    target_summary['Count'] / len(df) * 100
).round(2)

target_summary

,Count,Percentage
Target,,
Dropout,1421,32.12
Enrolled,794,17.95
Graduate,2209,49.93


## A4. Chosen evaluation metric

The primary evaluation metric is the **Macro-Averaged F1-Score (Macro-F1)**. The dataset has imbalanced target classes: `Graduate`, `Dropout`, and `Enrolled`. Macro-F1 calculates an F1-score for each class and averages the three scores, giving each class equal importance.

Per-class **precision**, **recall**, and a **confusion matrix** will also be examined.

## A7. Naive baseline

The baseline uses the **majority-class strategy**. It predicts the most frequent category in the training data. In the complete dataset, the most frequent category is `Graduate` (2,209 of 4,424 records, or approximately 49.93%).

The project proposal estimates a Macro-F1 score of approximately **0.222** for this baseline. The code below calculates its actual score using stratified cross-validation. The majority category is determined from each training fold only.

**Cross-validation setting:** The project proposal does not specify a fold count. This notebook uses 5-fold stratified cross-validation as the calculation setting.

In [ ]:
# Import the baseline model and cross-validation tools
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

# Separate predictor features (X) and target labels (y)
X = df.drop(columns=['Target'])
y = df['Target']

# Set up 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create a majority-class baseline
baseline = DummyClassifier(strategy='most_frequent')

# Calculate Macro-F1 for each fold
fold_macro_f1 = cross_val_score(
    baseline,
    X,
    y,
    cv=cv,
    scoring='f1_macro'
)

print('Macro-F1 for each fold:', [round(score, 4) for score in fold_macro_f1])
print(f'Mean cross-validation Macro-F1: {fold_macro_f1.mean():.4f}')

Macro-F1 for each fold: [np.float64(0.2217), np.float64(0.2221), np.float64(0.2221), np.float64(0.2221), np.float64(0.2222)]
Mean cross-validation Macro-F1: 0.2220


In [ ]:
# Generate out-of-fold predictions for the baseline
predicted_labels = cross_val_predict(baseline, X, y, cv=cv)

# Calculate precision, recall, F1-score, and support for each class
precision, recall, f1, support = precision_recall_fscore_support(
    y,
    predicted_labels,
    labels=class_labels,
    zero_division=0
)

per_class_results = pd.DataFrame(
    {
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'Support': support
    },
    index=class_labels
)

per_class_results.index.name = 'Target class'
per_class_results

,Precision,Recall,F1-score,Support
Target class,,,,
Dropout,0.000000,0.0,0.000000,1421
Enrolled,0.000000,0.0,0.000000,794
Graduate,0.499322,1.0,0.666064,2209


In [ ]:
# Display the baseline confusion matrix
matrix = confusion_matrix(y, predicted_labels, labels=class_labels)

confusion_matrix_table = pd.DataFrame(
    matrix,
    index=[f'Actual {label}' for label in class_labels],
    columns=[f'Predicted {label}' for label in class_labels]
)

confusion_matrix_table

,Predicted Dropout,Predicted Enrolled,Predicted Graduate
Actual Dropout,0,0,1421
Actual Enrolled,0,0,794
Actual Graduate,0,0,2209


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful